# 第01章 OpenCV入门（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 1 章的跟随代码。
> **全量收录原书代码**；原 notebook 缺失的例题（例 1.1–1.7）已按书补全，内容依据站点笔记 ch01.md。
> `cv2.imshow` 需在**本地 Python 环境**运行（弹窗显示图像，网页预览无法显示）；**工作目录须为「数字图像处理-python」根目录**（picture1.jpg 等素材都在这里），否则 imread 返回 None。

## 1.1 如何使用：Python、Anaconda 与 OpenCV 的安装

环境配置三步：装 Python 3 → 装 Anaconda → 装 OpenCV。OpenCV 用 PyPI 预编译包安装即可：

```text
pip install opencv-python           # 只装主库（本书大部分章节够用）
pip install opencv-contrib-python   # 主库 + 贡献库，二选一，不要同时装
```

注意：模块名是 `cv2` 而不是 `opencv`，`import opencv` 会直接报错；cv2 指"引入改善 API（面向对象）的模块"，并非"OpenCV 2 专用"。

In [ ]:
# ===== 1.1 验证安装：能 import、能报版本，就说明装好了 =====
import cv2
print(cv2.__version__)   # 例如 4.10.0；报 ModuleNotFoundError 才是没装上

## 1.2 图像处理基本操作

读取图像、显示图像、保存图像是最基本的三个操作，本章把"读 → 显 → 存"这条最小闭环完整走通：

- 读取图像 `cv2.imread(filename[, flags])`（§1.2.1，例 1.1）
- 显示图像 `cv2.imshow()` + `cv2.waitKey()`（§1.2.2，例 1.2–1.6）
- 保存图像 `cv2.imwrite()`（§1.2.3，例 1.7）

### 1.2.1 读取图像：cv2.imread()

`retval = cv2.imread(filename[, flags])`——读取成功返回 numpy 数组（BGR 顺序），**读不到只返回 None，不报错**。

- `filename`：完整文件名（路径名 + 文件名），相对路径按**当前工作目录**解析；
- `flags`：读取标记。最常用三个：`cv2.IMREAD_UNCHANGED`（−1，保持原格式）、`cv2.IMREAD_GRAYSCALE`（0，转单通道灰度）、`cv2.IMREAD_COLOR`（1，默认，3 通道 BGR）。

### 例 1.1 · 读取图像并打印

图像就是数组，`shape` = (高, 宽, 通道)，比 `print(img)` 实用；读取后先做 None 检查。

In [ ]:
# ===== 1.2.1 例 1.1：读取图像 =====
import cv2

img = cv2.imread("picture1.jpg")    # 默认 flags=1：3 通道 BGR 彩色图
print(type(img))                    # <class 'numpy.ndarray'>：图像就是数组
print(img.shape)                    # (1080, 1920, 3)：高 1080、宽 1920、3 通道 BGR
# print(img)                        # 能打印，但满屏数字没法看——先看 shape

# 读不到不报错、只返回 None，务必养成检查习惯：
if img is None:
    print("读取失败：检查文件名与当前工作目录")

In [ ]:
# ===== 1.2.1 flags：同一张图，三种读法 =====
import cv2

img_u = cv2.imread("picture1.jpg", -1)                     # −1：保持原格式
img_g = cv2.imread("picture1.jpg", cv2.IMREAD_GRAYSCALE)   # 0：单通道灰度
img_c = cv2.imread("picture1.jpg", cv2.IMREAD_COLOR)       # 1：3 通道 BGR（默认）

print(img_u.shape)   # (1080, 1920, 3)
print(img_g.shape)   # (1080, 1920)      ← 灰度图没有第 3 维
print(img_c.shape)   # (1080, 1920, 3)

### 1.2.2 显示图像：imshow、waitKey 与窗口管理

显示相关的函数有五个：`namedWindow`（建窗）、`imshow`（显示）、`waitKey`（等待按键）、`destroyWindow`（销毁指定窗口）、`destroyAllWindows`（销毁所有窗口）。本节含例 1.2–1.6 与 cv_show 封装。

高频报错预警：`error: (-215:Assertion failed) size.width>0 && size.height>0` 说明**要显示的图像是 None**——根源几乎总在读取环节（文件名写错或工作目录不对），用例 1.1 的 `if img is None` 检查当场现形。

### 例 1.2 · 先建窗再显示

- `None = cv2.namedWindow(winname)`：创建指定名称的窗口；
- `None = cv2.imshow(winname, mat)`：在指定窗口内显示图像。

可以先 `namedWindow` 建窗、再 `imshow` 引用它。

In [ ]:
# ===== 1.2.2 例 1.2：先建窗再显示 =====
import cv2

img = cv2.imread("picture1.jpg")
cv2.namedWindow("lesson")       # 创建名为 lesson 的窗口
cv2.imshow("lesson", img)       # 在 lesson 窗口内显示图像
cv2.waitKey(0)                  # 等待按键（见下文，没有它窗口画不出来）
cv2.destroyAllWindows()         # 按键后销毁窗口，程序干净收尾

实际使用中**通常省略 namedWindow**：直接让 `imshow` 引用一个不存在的窗口，它会自动"创建窗口 + 显示图像"两步完成。下面这个 cell 就是省事写法，末尾顺便定义了 `cv_show(name, img)`（显示一幅图、按任意键后销毁窗口再返回），供后续 cell 复用。

In [1]:
import cv2
img=cv2.imread("picture1.jpg")
cv2.imshow("Image",img)
cv2.waitKey(0)
cv2.destroyAllWindows()
def cv_show(name,img):
    cv2.imshow(name,img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()



### 例 1.3 · waitKey 返回 ASCII 码，做键盘交互

- `retval = cv2.waitKey([delay])`：等待按键。delay 单位是 **ms**，负数或 0（默认）表示无限等待；
- 返回值：无按键返回 −1，有按键返回该按键的 ASCII 码；
- 判断特定键用 `ord()` 取字符的 ASCII 码再比较（如 `key == ord('A')`），不用查表。

waitKey 既是"暂停程序"，又是 GUI 事件泵——没有它，imshow 的窗口会一闪而过甚至"未响应"。

In [ ]:
# ===== 1.2.2 例 1.3：waitKey 返回 ASCII 码，做键盘交互 =====
import cv2

img = cv2.imread("picture1.jpg")
cv2.imshow("demo", img)
key = cv2.waitKey()              # 默认 0：无限等待，按下的键返回其 ASCII 码
if key == ord('A'):              # ord('A') = 65：不查 ASCII 表也能写分支
    cv2.imshow("PressA", img)    # 按 A：新开窗口 PressA
elif key == ord('B'):
    cv2.imshow("PressB", img)    # 按 B：新开窗口 PressB
# 原书示例到此为止；不加下面两行，新窗口会随脚本结束一闪而过：
cv2.waitKey()
cv2.destroyAllWindows()

### 例 1.4 · waitKey 充当暂停键

从"暂停"的角度使用 waitKey：程序停在 `cv2.waitKey()` 这一行直到有按键被按下，随后返回值不为 −1、继续往下执行。

In [ ]:
# ===== 1.2.2 例 1.4：waitKey 充当暂停键 =====
import cv2

img = cv2.imread("picture1.jpg")
cv2.imshow("demo", img)
key = cv2.waitKey()       # 未按键：程序停在这一行；按键：往下执行
if key != -1:             # 有按键时返回值不是 -1
    print("触发了按键")

### 例 1.5 / 例 1.6 · 销毁窗口

- `None = cv2.destroyWindow(winname)`：释放指定名称的窗口；
- `None = cv2.destroyAllWindows()`：释放所有窗口（更常用）。

两个窗口显示同一幅图，按键后统一销毁收尾。

In [ ]:
# ===== 1.2.2 例 1.5 / 1.6：销毁窗口 =====
import cv2

img = cv2.imread("picture1.jpg")
cv2.imshow("demo1", img)
cv2.imshow("demo2", img)          # 两个窗口显示同一幅图
cv2.waitKey()                     # 等待按键
cv2.destroyWindow("demo1")        # 只销毁 demo1
# cv2.destroyAllWindows()         # 或一键销毁全部（更常用，见例 1.2/1.3）

### 1.2.3 保存图像：cv2.imwrite()

`retval = cv2.imwrite(filename, img[, params])`——保存成功返回 True，失败返回 False。filename 要**包含文件扩展名**，扩展名决定按什么格式编码保存；params 为可选的保存类型参数。

### 例 1.7 · 读 → 显 → 存 完整闭环

把三步串起来，再用 flags=0 演示灰度链：读取时转灰度 → 显示 → 另存为新文件。

In [ ]:
# ===== 1.2.3 例 1.7：读 → 显 → 存 完整闭环 =====
import cv2

# ① 读取（工作目录：数字图像处理-python 根目录）
img = cv2.imread("picture1.jpg")

# ② 显示：imshow + waitKey + destroyAllWindows 三件套
cv2.imshow("Image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# ③ 保存：扩展名 .jpg 决定按 JPEG 编码，成功返回 True
r = cv2.imwrite("result.jpg", img)
print(r)                                             # True

In [ ]:
# 灰度版：读取时直接转灰度（flags=0）→ 显示 → 另存
img_gray = cv2.imread("picture1.jpg", cv2.IMREAD_GRAYSCALE)
cv2.imshow("Gray Image", img_gray)
cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.imwrite("picture1_gray.jpg", img_gray)           # 生成 picture1_gray.jpg